NAT: Network address translation
If the private IP of the endpoints are used to communicate directly over internet it may pose security threat to that device. To safeguard my endpoints I'll translate my endpoint's private IP to firewall's public IP. So, here I'll be applying my knowledge of SNAT and DNAT.

In [32]:
import random;

Router_IP = "155.26.24.23" #Static IP address for internet facing router
address_book = {};
active_endpoints = [];

def random_private_ip(router_IP):
    # Generate a random private IP address in the range other than the router IP
    private_ip = "192.168.1." + str(random.randint(1,10))
    #if private_ip != router_IP:
    if private_ip not in active_endpoints:        
        active_endpoints.append(private_ip)
        return private_ip  + ":" + str(random.randint(1,1023))
    return random_private_ip(router_IP)  # Recursively call until a different IP is generated

def generate_router_port():
    # Generate a random port number between 1024 and 65535 for router
    port_number = random.randint(1024, 1074)
    if port_number in address_book.keys():
        print("Port number already exists in address book. Generating a new port number...")
        return generate_router_port()  # Recursively call until a unique port number is generated
    return port_number

def router_address_book(IPAddress, connection_type):
    #generate new port number and add to address book
    if connection_type == "outbound":
        router_port_number = generate_router_port()
        address_book[router_port_number] = IPAddress
        return router_port_number
        

def network_address_translation(IPAddress):
    translated_address = Router_IP + ":" + str(router_address_book(IPAddress, connection_type = "outbound"))
    return translated_address

def main():
     
    ####Source Network Address Translation (SNAT) - Convert private IP addresses to public IP address

    for counter in range(5):  # Generate 5 random private IP addresses and convert them
        private_ip = random_private_ip(Router_IP)
        converted_ip = network_address_translation(private_ip)
        print(counter, ")", "Private IP:", private_ip, ", ", "Converted IP Address:", converted_ip, ", ", "Connection Type: Outbound", "\n")
        #print("Converted IP Address:", converted_ip)

    ####END of Source Network Address Translation (SNAT) - Convert private IP addresses to public IP address
    
    print("Address Book:", address_book)
    print("Active Endpoints:", active_endpoints)


if __name__ == "__main__":
    main()


0 ) Private IP: 192.168.1.2:273 ,  Converted IP Address: 155.26.24.23:1032 ,  Connection Type: Outbound 

1 ) Private IP: 192.168.1.9:721 ,  Converted IP Address: 155.26.24.23:1044 ,  Connection Type: Outbound 

2 ) Private IP: 192.168.1.6:382 ,  Converted IP Address: 155.26.24.23:1025 ,  Connection Type: Outbound 

3 ) Private IP: 192.168.1.3:12 ,  Converted IP Address: 155.26.24.23:1061 ,  Connection Type: Outbound 

4 ) Private IP: 192.168.1.7:138 ,  Converted IP Address: 155.26.24.23:1070 ,  Connection Type: Outbound 

Address Book: {1032: '192.168.1.2:273', 1044: '192.168.1.9:721', 1025: '192.168.1.6:382', 1061: '192.168.1.3:12', 1070: '192.168.1.7:138'}
Active Endpoints: ['192.168.1.2', '192.168.1.9', '192.168.1.6', '192.168.1.3', '192.168.1.7']


In [36]:
import random;

Router_IP = "155.26.24.23" #Static IP address for internet facing router
address_book = {};
active_endpoints = [];

def random_private_ip(router_IP):
    # Generate a random private IP address in the range other than the router IP
    private_ip = "192.168.1." + str(random.randint(1,10))
    #if private_ip != router_IP:
    if private_ip not in active_endpoints:        
        active_endpoints.append(private_ip)
        return private_ip  + ":" + str(random.randint(1,1023))
    return random_private_ip(router_IP)  # Recursively call until a different IP is generated

def generate_router_port():
    # Generate a random port number between 1024 and 65535 for router
    port_number = random.randint(1024, 1074)
    if port_number in address_book.keys():
        print("Port number already exists in address book. Generating a new port number...")
        return generate_router_port()  # Recursively call until a unique port number is generated
    return port_number

def router_address_book(IPAddress, connection_type):
    
    if connection_type == "outbound":   #generate new port number and add to address book
        router_port_number = generate_router_port()
        address_book[router_port_number] = IPAddress
        return router_port_number

    elif connection_type == "inbound":
        router_port_number = int(IPAddress.split(":")[1])  # Extract the port number from the inbound IP address)

        if router_port_number not in address_book.keys():
            return IPAddress

        if int(router_port_number) in address_book.keys():
            return address_book[router_port_number]

def network_address_translation(IPAddress,connection_type):
    if connection_type == "outbound":
        translated_address = Router_IP + ":" + str(router_address_book(IPAddress, connection_type))
    
    elif connection_type == "inbound":
        translated_address = router_address_book(IPAddress, connection_type)

    return translated_address

def main():
     
    ####Source Network Address Translation (SNAT) - Convert private IP addresses to public IP address

    for counter in range(5):  # Generate 5 random private IP addresses and convert them
        private_ip = random_private_ip(Router_IP)
        converted_ip = network_address_translation(private_ip,"outbound")
        print(counter, ")", "Private IP:", private_ip, ", ", "Converted IP Address:", converted_ip, ", ", "Connection Type: Outbound", "\n")
        #print("Converted IP Address:", converted_ip)

    ####END of Source Network Address Translation (SNAT) - Convert private IP addresses to public IP address

    print("Address Book:", address_book)
    print("Active Endpoints:", active_endpoints, "\n")

    ###Destination Network Address Translation (DNAT) - Convert public IP addresses to private IP address
    for counter in range(5):
        inbound_ip = Router_IP + ":" + str(random.randint(1024, 1074))#generating an inbound IP address with random port number
        converted_ip = network_address_translation(inbound_ip,"inbound")
        if converted_ip == inbound_ip:
            print("Port number not found in address book. Unable to convert inbound IP address:", inbound_ip, "\n")
        else:
            print(counter, ")", "Inbound IP:", inbound_ip, ", ", "Converted IP Address:", converted_ip, ", ", "Connection Type: Inbound", "\n")


if __name__ == "__main__":
    main()


0 ) Private IP: 192.168.1.4:782 ,  Converted IP Address: 155.26.24.23:1037 ,  Connection Type: Outbound 

1 ) Private IP: 192.168.1.1:265 ,  Converted IP Address: 155.26.24.23:1038 ,  Connection Type: Outbound 

2 ) Private IP: 192.168.1.3:458 ,  Converted IP Address: 155.26.24.23:1032 ,  Connection Type: Outbound 

3 ) Private IP: 192.168.1.6:879 ,  Converted IP Address: 155.26.24.23:1056 ,  Connection Type: Outbound 

4 ) Private IP: 192.168.1.10:391 ,  Converted IP Address: 155.26.24.23:1048 ,  Connection Type: Outbound 

Address Book: {1037: '192.168.1.4:782', 1038: '192.168.1.1:265', 1032: '192.168.1.3:458', 1056: '192.168.1.6:879', 1048: '192.168.1.10:391'}
Active Endpoints: ['192.168.1.4', '192.168.1.1', '192.168.1.3', '192.168.1.6', '192.168.1.10'] 

0 ) Inbound IP: 155.26.24.23:1048 ,  Converted IP Address: 192.168.1.10:391 ,  Connection Type: Inbound 

Port number not found in address book. Unable to convert inbound IP address: 155.26.24.23:1041 

Port number not found in add